# Part 02: Collections and Loops

**Time needed:** 4-5 hours | **Prerequisite:** Part 01

Run each cell with **Shift+Enter**, read the output, then **change the code and run it again**.

Select your `.venv` kernel first (top right of the notebook).

## Why this part matters

In Part 01 every variable held **one** value. Real AI work is about **groups** of things:

| In AI engineering you handle... | ...which is stored as a |
|---|---|
| a conversation (many messages) | **list** |
| one message (`role` + `content`) | **dictionary** |
| token usage `(input, output)` | **tuple** |
| the unique labels in a dataset | **set** |
| "do this for every document" | **loop** |

By the end you'll be able to read the data an LLM API sends back and process a whole conversation with a few lines of code.

## What we'll cover

1. Lists
2. Tuples
3. Dictionaries
4. Sets
5. Loops: `for`, `while`, `range`, `enumerate`, `zip`
6. Comprehensions
7. Putting it together: trimming a chat history to fit a token budget

## 1. Lists

A **list** is an ordered collection of items, written in square brackets. Items can be added, removed and changed.

In [25]:
prompts = ["summaries", "translate", "classify"]

print(prompts)
print(type(prompts))
print(len(prompts))

['summaries', 'translate', 'classify']
<class 'list'>
3


### Indexing and slicing (just like strings!)

Positions start at **0**. Negative numbers count from the end. A slice `[start:stop]` goes up to, but **not including**, `stop`.

```
index:    0           1            2
        "summarise" "translate" "classify"
negative: -3          -2           -1
```

In [26]:
print(prompts[0])           # -------> This would print the first item
print(prompts[-1])          # -------> This would print the last item
print(prompts[0:2])         # -------> This would print items at the 0 and 1 position
print(prompts[1:])          # -------> This would print the item from index 1 to the end
print(prompts[:2])          # -------> This would print items from the start up to (not including) index 2



summaries
classify
['summaries', 'translate']
['translate', 'classify']
['summaries', 'translate']


### Changing a list

Lists are **mutable**: meaning that they can be changed in place. (Strings were *immutable*, remember?)

In [27]:
prompts[0] = "summarize"      # --------> replacing the item at index 0 from "summaries" to "summarize"
prompts.append("extract")     # --------> add to the END
prompts.insert(1, "rewrite")  # --------> This mean insert "rewrite" at position 1, shifting the rest along
print(prompts)

['summarize', 'rewrite', 'translate', 'classify', 'extract']


In [28]:
prompts.remove("classify")          # --------> Here we can remove an item by its VALUE (the first match)
print(prompts)

['summarize', 'rewrite', 'translate', 'extract']


In [29]:
last = prompts.pop()
print("popped:", last)
print(prompts)

popped: extract
['summarize', 'rewrite', 'translate']


### Useful list functions

Here is a list of token counts, one per request. These built-in functions work on any list of numbers:

In [30]:
token_counts = [120, 45, 300, 80]

print("how many   :", len(token_counts))
print("total      :", sum(token_counts))
print("smallest   :", min(token_counts))
print("largest    :", max(token_counts))
print("average    :", sum(token_counts) / len(token_counts))
print("is 45 in it:", 45 in token_counts)       # `in` works on lists too

how many   : 4
total      : 545
smallest   : 45
largest    : 300
average    : 136.25
is 45 in it: True


### Sorting: `sorted()` vs `.sort()`

- `sorted(x)` returns a **new** sorted list and leaves the original alone
- `x.sort()` sorts the list **in place** and returns `None`

In [31]:
token_counts = [120, 45, 300, 80]

print(sorted(token_counts))
print(sorted(token_counts, reverse=True))
print(token_counts)


[45, 80, 120, 300]
[300, 120, 80, 45]
[120, 45, 300, 80]


In [32]:
token_counts.sort()             # -------> .sort() Changes the list itself
print(token_counts)

[45, 80, 120, 300]


> **Classic bug:** `token_counts = token_counts.sort()` sets the variable to `None`, because `.sort()` returns nothing. Either call `.sort()` on its own line or use `sorted()`.

### Lists can hold anything, even other lists

An **embedding** (a way to turn text into numbers so a model can compare meanings) is just a list of floats. Real ones have hundreds or thousands of numbers; here are three:

In [33]:
embedding = [0.12, -0.45, 0.88]
print(len(embedding), "dimensions")

3 dimensions


In [34]:
# A list of lists: one embedding per document
document_embeddings = [
    [0.12, -0.45, 0.88],
    [0.50, 0.10, -0.20],
]
print(document_embeddings[1])               # ----------> this prints the second embedding
print(document_embeddings[1][0])            # ----------> this prints the first number of the second embedding

[0.5, 0.1, -0.2]
0.5


### Gotcha: copying a list

`b = a` does **not** copy a list. Both names point at the **same** list:

In [35]:
original = ["a", "b"]
alias = original                # NOT a copy: it's just a second name for the same list

alias.append("c")
print(alias)
print(original)                 # It's surprising to see "original changes too!"


['a', 'b', 'c']
['a', 'b', 'c']


In [36]:
safe = original.copy() # a real, independent copy
safe.append("d")
print(original)         # unchanges by the copy
print(safe)

['a', 'b', 'c']
['a', 'b', 'c', 'd']


---
## 2. Tuples

A **tuple** is like a list that **cannot be changed** after creation. It uses round brackets. Use one when the data is a fixed bundle: *"these two numbers always go together."*

APIs often report token usage this way: `(input_tokens, output_tokens)`.

In [37]:
usage = (1200, 350)

print(usage)
print(type(usage))
print(usage[0])                 # Indexing works same way with list
print(usage[-1])
print(len(usage))

(1200, 350)
<class 'tuple'>
1200
350
2


Trying to change one raises an error:

```python
usage[0] = 999     # TypeError: 'tuple' object does not support item assignment
```

### Unpacking: the best thing about tuples

You can split a tuple into separate variables in one line. The number of names must match the number of items:

In [38]:
usage = (1200, 350)

input_tokens, output_tokens = usage                 # first item -> first name, second -> second name
print("input :", input_tokens)
print("output:", output_tokens)
print("total :", input_tokens + output_tokens)

# swapping two variables is a one-liner. All thanks to tuples
a, b = 1, 2
a, b = b, a
print(a, b)

input : 1200
output: 350
total : 1550
2 1


### Gotcha: a one-item tuple needs a trailing comma

Round brackets alone are just maths grouping:


In [39]:
print(type((5)))            # int: just the number 5 in brackets
print(type((5,)))           # tuple: the comma makes it one

<class 'int'>
<class 'tuple'>


### List of tuples: a very common shape

You'll often loop over pairs like this. Notice we unpack each pair directly in the `for` line:

In [40]:
word_counts = [("token", 4), ("cost", 3), ("model", 2)]

for word, count in word_counts:
    print(f"{word} appears {count} times")

token appears 4 times
cost appears 3 times
model appears 2 times


---
## 3. Dictionaries

A **dictionary** (`dict`) stores **key: value** pairs, like a real dictionary maps a word to its meaning. You look things up by **key** instead of by position.

This is **the** most important data structure for AI engineering. API requests, API responses, chat messages, configuration and JSON are all dictionaries.

In [41]:
request = {
    "model": "example-model",
    "temperature": 0.7,
    "max_tokens": 256,
    "stream": False,

}

print(request["model"])             # you can call a value by its key
print(request["max_tokens"])
print(len(request))

example-model
256
4


### Safe lookup with `.get()`

Asking for a key that doesn't exist with `[]` crashes your program (`KeyError`). `.get()` returns `None`, or a default you choose, instead:

In [42]:
print(request.get("top_p"))           # None: key is missing, no crash
print(request.get("top_p", 1.0))      # 1.0: our own default
print(request.get("model", "none"))   # the key exists, so the real value wins

print("temperature" in request)       # `in` checks KEYS
print("top_p" in request)

None
1.0
example-model
True
False


### Adding, changing and deleting in a dictionary

In [43]:
request["top_p"] = 0.9          # I just added this new key known as top_p
request["temperature"] = 0.2    # existing key: assigning replaces the value
del request["stream"]           # This is to delete a key
print(request)

{'model': 'example-model', 'temperature': 0.2, 'max_tokens': 256, 'top_p': 0.9}


### Keys, values and items

You'll use these three constantly when looping:

In [44]:
print(list(request.keys()))         # all keys
print(list(request.values()))       # all values
print(list(request.items()))        # (key, value) tuples


['model', 'temperature', 'max_tokens', 'top_p']
['example-model', 0.2, 256, 0.9]
[('model', 'example-model'), ('temperature', 0.2), ('max_tokens', 256), ('top_p', 0.9)]


### Merging dictionaries (default settings + your overrides)

This is a common pattern for configuration: start with defaults, and layer your own settings on top. The `|` operator merges two dicts, and **the right-hand side wins** on conflicts:

In [45]:
defaults = {"temperature": 0.7, "max_tokens": 256}
overrides = {"temperature": 0.1}

settings = defaults | overrides
print(settings)

{'temperature': 0.1, 'max_tokens': 256}


### Nested data: dictionaries inside lists inside dictionaries

Real API responses are nested. Read them **from the outside in**, one key or index at a time. Here is a simplified response, similar in shape to what LLM APIs send back:

In [47]:
response = {
    "id": "resp_001",
    "model": "example-model",
    "choices": [
        {"role": "assistant", "content": "tokens are small chunks of text."}
    ],
    "usage": {"input_tokens": 12, "output_tokens": 8},
}

choices = response["choices"]           # -----> This would be a list
first_choice = choices[0]              # -----> This would be a dictionary
text = first_choice["content"]          # -----> a string
print(text)

# The same thing in one step: read left to right
print(response["choices"][0]["content"])
print(response["usage"]["output_tokens"])

tokens are small chunks of text.
tokens are small chunks of text.
8


### A conversation is a list of dictionaries

This is **the format chat APIs use for message history**. One dict per message. Memorise this shape, because you'll see it for the rest of the course:

In [48]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is a token?"},
    {"role": "assistant", "content": "A token is a small chunk of text."},
    {"role": "user", "content": "Why does it matter?"},
]

print(len(conversation), "messages")
print(conversation[1])                      # a whole message (a dict)
print(conversation[1]["content"])           # one value from that message
print(conversation[-1]["role"])             # role of the newest message

4 messages
{'role': 'user', 'content': 'What is a token?'}
What is a token?
user


---
## 4. Sets

A **set** is a collection of **unique** items with **no order**. It's written with curly brackets but **without** key: value pairs. Sets are perfect for removing duplicates and for fast "is this in the collection?" checks.

In [49]:
tags = ["ai", "python", "ai", "nlp", "python", "ai"]

unique_tags = set(tags)         # this would remove duplicates
print(sorted(unique_tags))      # sets have no order, so we sort just for display
print(len(tags), "->", len(unique_tags))

['ai', 'nlp', 'python']
6 -> 3


### Adding, removing and checking

In [50]:
labels = {"positive", "negative"}

labels.add("neutral")           # add one item
labels.add("positive")          # already there: nothing happens (no error, no duplicate)
labels.discard("negative")      # remove if present (no error if it isn't)

print(sorted(labels))
print("neutral" in labels)      # very fast, even for millions of items

['neutral', 'positive']
True


> **Take note:** `{}` creates an empty **dictionary**, not an empty set. Use `set()` for an empty set.

In [51]:
print(type({}))
print(type(set()))

<class 'dict'>
<class 'set'>


### Set maths: `&` (both), `|` (either), `-` (difference)

Two documents have these tags:

In [52]:
doc_a = {"ai", "python", "nlp", "llm"}
doc_b = {"python", "llm", "data"}

print("in both     :", sorted(doc_a & doc_b))
print("in either   :", sorted(doc_a | doc_b))
print("only in A   :", sorted(doc_a - doc_b))
print("only in B   :", sorted(doc_b - doc_a))

in both     : ['llm', 'python']
in either   : ['ai', 'data', 'llm', 'nlp', 'python']
only in A   : ['ai', 'nlp']
only in B   : ['data']


### Where sets show up in AI work

- **Stop words**: a set of very common words to ignore when counting words
- **Deduplicating** documents or labels before processing
- **Vocabulary size**: `len(set(words))` is the number of *different* words in a text

In [53]:
text = "the model reads the text and the model writes the reply"
words = text.split()

print("total words      :", len(words))
print("different words:", len(set(words)))

total words      : 11
different words: 7


---
## 5. Loops

A **loop** repeats code. AI work is full of "do this for every document / message / row". Two kinds:

- `for` loop: repeat for **each item** in a collection
- `while` loop: repeat **as long as** a condition is true

### The `for` loop

In [54]:
prompts = ["summarise", "translate", "classify"]

for prompt in prompts:
    print("Running task:", prompt)

Running task: summarise
Running task: translate
Running task: classify


**Line by line:**

- `for prompt in prompts:` means *"take the items one at a time; call the current one `prompt`"*
- The **colon** ends the line, and the **indented** lines below are the loop body, repeated once per item
- `prompt` is just a variable name. Pick one that reads well (`for message in messages`, `for doc in documents`)

### `range()`: loop a number of times

In [55]:
for i in range(3):                  # 0, 1, 2  (stops BEFORE 3)
    print("attempt", 1)

print(list(range(2, 8)))            # start at 2, stop before 8
print(list(range(0, 10, 3)))        # start, stop, STEP: 0, 3, 6, 9

attempt 1
attempt 1
attempt 1
[2, 3, 4, 5, 6, 7]
[0, 3, 6, 9]


### `enumerate()`: get the position AND the item

Handy for numbering things. `start=1` makes the counter begin at 1 instead of 0:

In [56]:
for number, prompt in enumerate(prompts, start=1):
    print(f"{number}. {prompt}")

1. summarise
2. translate
3. classify


### `zip()`: loop over two lists side by side

In [57]:
documents = ["report.pdf", "notes.txt", "email.eml"]
token_counts = [4200, 310, 95]

for name, tokens in zip(documents, token_counts):
    print(f"{name:<12} {tokens:>5} tokens")

report.pdf    4200 tokens
notes.txt      310 tokens
email.eml       95 tokens


(`{name:<12}` pads text to 12 characters, left-aligned; `{tokens:>5}` right-aligns in 5. Format specs again, from Part 01.)

### Looping over a dictionary

Use `.items()` to get each key and value together:

In [60]:
request = {"model": "example-model", "temperature": 0.2, "max_tokens": 256}

for key, value in request.items():
    print(f"{key} = {value}")

model = example-model
temperature = 0.2
max_tokens = 256


### Pattern 1: accumulate (add things up)

Start with an empty "total" and add to it inside the loop:

In [61]:
conversation = [
    {"role": "user", "tokens": 12},
    {"role": "assistant", "tokens": 85},
    {"role": "user", "tokens": 9},
    {"role": "assistant", "tokens": 120},
]

total = 0                       # start at zero BEFORE the loop
for message in conversation:
    total += message["tokens"]  # add each message's tokens

print("Total tokens:", total)

Total tokens: 226


### Pattern 2: filter (keep only some items)

Combine a loop with an `if`, and build a new list with `.append()`:

In [62]:
user_messages = []              # start with an empty list
for message in conversation:
    if message["role"] == "user":
        user_messages.append(message)

print(len(user_messages), "user messages")

2 user messages


### Pattern 3: count with a dictionary

This is the most useful pattern in the whole part. To count how often each word appears:

1. start with an empty dict
2. for each word, read its current count with `.get(word, 0)` (zero if we haven't seen it yet)
3. add 1 and store it back

In [63]:
text = "the cat and the dog and the bird"

counts = {}
for word in text.split():
    counts[word] = counts.get(word, 0) + 1

print(counts)

{'the': 3, 'cat': 1, 'and': 2, 'dog': 1, 'bird': 1}


Python also has a ready-made shortcut for exactly this, in the `collections` module. It's good to know, but **understand the pattern above first**, because the same idea powers many other tasks:

In [64]:
from collections import Counter

print(Counter(text.split()))
print(Counter(text.split()).most_common(2))     # the 2 most frequent

Counter({'the': 3, 'and': 2, 'cat': 1, 'dog': 1, 'bird': 1})
[('the', 3), ('and', 2)]


### `break` and `continue`

- `break` stops the loop completely
- `continue` skips the rest of this round and jumps to the next item

In [65]:
messages = ["Hello", "", "What is AI?", "STOP", "Never reached"]

for message in messages:
    if message == "":
        continue                 # skip empty messages
    if message == "STOP":
        break                    # leave the loop entirely
    print("Processing:", message)

Processing: Hello
Processing: What is AI?


### Batching: process a long list in chunks

APIs often limit how many items you can send at once, so you split the work into **batches**. The trick: `range` with a step, plus a slice. (Remember: slicing past the end of a list is safe.)

In [66]:
documents = ["d1", "d2", "d3", "d4", "d5", "d6", "d7", "d8", "d9", "d10"]
batch_size = 4

for start in range(0, len(documents), batch_size):    # start = 0, 4, 8
    batch = documents[start:start + batch_size]
    print(f"batch starting at {start}: {batch}")

batch starting at 0: ['d1', 'd2', 'd3', 'd4']
batch starting at 4: ['d5', 'd6', 'd7', 'd8']
batch starting at 8: ['d9', 'd10']


### The `while` loop

A `while` loop keeps going **while a condition is True**. Use it when you don't know in advance how many rounds you need, such as **retrying a failed API call**. Here we simulate calls that fail twice and then succeed. (In Part 06 you'll do this for real.)

In [67]:
simulated_results = ["error", "error", "ok"]    # what each attempt will return
MAX_ATTEMPTS = 5

attempt = 0
succeeded = False

while attempt < MAX_ATTEMPTS and not succeeded:
    result = simulated_results[attempt]
    attempt += 1                                 # count this attempt
    print(f"Attempt {attempt}: {result}")
    if result == "ok":
        succeeded = True

print("Succeeded:", succeeded, "after", attempt, "attempts")

Attempt 1: error
Attempt 2: error
Attempt 3: ok
Succeeded: True after 3 attempts


> **Danger: infinite loops.** If the condition never becomes False, the loop runs forever. Always make sure something inside the loop changes (here, `attempt += 1`) and keep a **maximum** like `MAX_ATTEMPTS` as a safety net. In a notebook, use the **Interrupt** button to stop a runaway loop.

### Nested loops: a loop inside a loop

Here is the inner loop running fully for every round of the outer loop. Useful for things like "for every message, look at every word":

In [68]:
chat = ["hello world", "what is AI"]

for message in chat:
    for word in message.split():
        print(f"{message!r:<14} -> {word}")

'hello world'  -> hello
'hello world'  -> world
'what is AI'   -> what
'what is AI'   -> is
'what is AI'   -> AI


---
## 6. Comprehensions

A **comprehension** builds a new collection from an old one in a single line. It replaces the "empty list + loop + append" pattern.

Compare. Doubling every token count with a loop:

In [69]:
token_counts = [120, 45, 300, 80]

doubled = []
for t in token_counts:
    doubled.append(t * 2)
print(doubled)

[240, 90, 600, 160]


...and as a comprehension:

In [70]:
doubled = [t * 2 for t in token_counts]
print(doubled)

[240, 90, 600, 160]


**How to read it:** `[ what_to_keep  for  item  in  collection ]`. Say it out loud: *"t times 2, for each t in token_counts."*

### Adding a filter

Put an `if` at the end to keep only some items:

In [71]:
big_requests = [t for t in token_counts if t > 100]
print(big_requests)

[120, 300]


### On a conversation

Remember the list-of-dicts shape? Pulling out just what you need is one line:

In [72]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is a token?"},
    {"role": "assistant", "content": "A token is a small chunk of text."},
    {"role": "user", "content": "Why does it matter?"},
]

user_texts = [m["content"] for m in conversation if m["role"] == "user"]
print(user_texts)

lengths = [len(m["content"]) for m in conversation]
print(lengths)

['What is a token?', 'Why does it matter?']
[28, 16, 33, 19]


### Dictionary and set comprehensions

Same idea, but with curly brackets. A dict comprehension uses `key: value`:

In [73]:
words = ["token", "model", "embedding"]

word_lengths = {word: len(word) for word in words}
print(word_lengths)

roles_used = {m["role"] for m in conversation}      # set comprehension: unique roles
print(sorted(roles_used))

{'token': 5, 'model': 5, 'embedding': 9}
['assistant', 'system', 'user']


### A one-line `if / else` inside a comprehension

When you want to **transform** items rather than drop them, put the `if / else` *before* the `for`:

In [74]:
sizes = ["long" if t > 100 else "short" for t in token_counts]
print(sizes)

['long', 'short', 'long', 'short']


> **Readability rule:** comprehensions are great for simple transformations and filters. If you need to stop and think about what a comprehension does, write the normal `for` loop instead. Clear code beats clever code.

---
## 7. Putting it all together: fit a chat history into a token budget

Models can only read a limited number of tokens, so chatbots **keep the newest messages and drop the oldest** once the history gets too long. (You'll build a full chatbot with memory in Part 09. This is the core idea.)

The plan:

1. Walk through the history **from newest to oldest** (`reversed()`)
2. Add each message while we still have budget
3. `break` as soon as the next message wouldn't fit
4. Insert each kept message at the **front** of a new list so the final order is oldest to newest again

Try changing `TOKEN_BUDGET` to see how many messages survive.

In [75]:
history = [
    {"role": "user", "content": "Hi, can you explain what a token is?"},
    {"role": "assistant", "content": "A token is a small piece of text, often part of a word."},
    {"role": "user", "content": "How many tokens are in a page of text?"},
    {"role": "assistant", "content": "A page of English is usually around 500 to 700 tokens."},
    {"role": "user", "content": "Does a longer history cost more?"},
    {"role": "assistant", "content": "Yes, every message you send again adds to the cost."},
]

TOKEN_BUDGET = 40          # try 20, 60, 200
CHARS_PER_TOKEN = 4

kept = []
used = 0

for message in reversed(history):                       # newest first
    tokens = max(1, len(message["content"]) // CHARS_PER_TOKEN)
    if used + tokens > TOKEN_BUDGET:
        break                                           # no room: stop here
    kept.insert(0, message)                             # put at the FRONT to keep order
    used += tokens

print(f"Kept {len(kept)} of {len(history)} messages, using ~{used} of {TOKEN_BUDGET} tokens\n")
for message in kept:
    print(f"{message['role']:<10} {message['content']}")

Kept 3 of 6 messages, using ~33 of 40 tokens

assistant  A page of English is usually around 500 to 700 tokens.
user       Does a longer history cost more?
assistant  Yes, every message you send again adds to the cost.


---
## Common mistakes

| Mistake | What happens | Fix |
|---|---|---|
| `my_list[3]` on a 3-item list | `IndexError` (valid indexes are 0 to 2) | remember counting starts at 0 |
| `my_dict["missing"]` | `KeyError` | use `.get("missing", default)` |
| `items = items.sort()` | `items` becomes `None` | call `.sort()` alone, or use `sorted()` |
| `b = a` for lists/dicts | both names share one list | use `a.copy()` |
| `{}` expecting an empty set | you get an empty dict | use `set()` |
| `for k, v in my_dict:` | error: unpacking text | use `my_dict.items()` |
| Forgetting to change the `while` condition | infinite loop | change something inside, add a max count |
| Removing items from a list while looping over it | items get skipped | loop over a copy, or build a new list |
| `(5)` instead of `(5,)` | an int, not a tuple | add the trailing comma |

## Recap

- **List** `[ ]`: ordered and changeable. Use for sequences (messages, documents, embeddings).
- **Tuple** `( )`: ordered and fixed. Use for bundles like `(input_tokens, output_tokens)`, and unpack them.
- **Dict** `{k: v}`: look up by key. Use for requests, responses and messages. Prefer `.get()` for safe lookups.
- **Set** `{ }`: unique items, no order. Use for deduplicating and fast membership checks.
- **`for`** loops over items; **`while`** loops until a condition changes. Use `range`, `enumerate`, `zip` and `.items()`.
- **Patterns:** accumulate, filter, count with a dict, batch with `range` + slice.
- **Comprehensions** are one-line versions of "build a new collection from an old one".
- **A conversation is a list of dicts** with `role` and `content`.

## Next steps

1. Do the practice in `exercises.py`. Run it with `python exercises.py`.
2. Build the **Chat Log Analyzer** in `mini_project/chat_log_analyzer.py`.
3. Compare with `solutions.py` only after you've tried.

## Your turn

In the cell below:

1. Make a list of at least 5 short messages (strings).
2. Use a loop and a dictionary to count how many messages have fewer than 20 characters and how many have 20 or more.
3. Use a list comprehension to build a list of the messages that contain the word `"AI"` (any capitalisation).